# Drawing with epicycles

*maths · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/fourier_epicycles)

A closed curve sampled as complex numbers $z_n = x_n + i\,y_n$ has a discrete Fourier transform $c_k$, and every term $c_k e^{2\pi i k t}$ is a circle of radius $|c_k|$ turning $k$ times per trip. Chain the circles tip to tail, biggest first, and the pen retraces the curve. Watch the outline of the Tensorizer logo appear as circles are added, while the error falls exactly as Parseval predicts.

**Vector form**

$$c_k = \frac1N\sum_{n=0}^{N-1} z_n\, e^{-2\pi i k n/N},\qquad z(t) = \sum_k c_k\, e^{2\pi i k t},\qquad \frac1N\sum_n |z_n|^2 = \sum_k |c_k|^2$$

**Index (tensor) form**

$$c_k = \tfrac1N\,W_{kn}\,z_n \qquad S_{mt} = K_{mk}\,c_k\,E_{tk} \qquad E_m = \tfrac1N\,|z_n - S_{mn}|^2 = \sum_{r_k \ge M_m} |c_k|^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $z_n$ | `shape(name, n, device)` | (N,) | the curve as complex numbers at equal time steps $t_n = n/N$; polygons are resampled by arc length |
| $W_{kn}$ | `exp(-2j * pi * outer(k, n) / N)` | (N, N) | the DFT matrix, frequencies $k = -N/2 \dots N/2 - 1$ |
| $c_k$ | `einsum('kn,n->k', W, z) / N` | (N,) | repeated $n$ ⇒ summed: circle $k$ has radius $/c_k/$ and starting angle $\arg c_k$ |
| $r_k$ | `argsort(c.abs(), descending=True)` | (N,) | circles ranked by radius: the chain keeps the biggest first |
| $S_{mt}$ | `einsum('mk,k,tk->mt', keep, c, E)` | (M, T) | repeated $k$ ⇒ summed: the curve drawn by the $M_m$ largest circles, for every $M$ at once ($K_{mk} = [r_k < M_m]$, $E_{tk} = e^{2\pi i k t}$) |
| $\text{centres}$ | `cumsum(c[order] * exp(2j * pi * k[order] * t), 1)` | (T, M + 1) | running sum of the terms: the centre of every circle in the chain, the last one is the pen |
| $E_m$ | `sample_error(z, S)` | (M,) | mean squared error at the samples, equal to the energy of the circles left out (Parseval) |

The DFT matrix $W$ writes the sum out in full ($N^2$ products); `torch.fft.fft` gives the same $c_k$ with the Cooley–Tukey fast Fourier transform in $O(N \log N)$, and a test checks they agree. The DFT is invertible: all $N$ circles reproduce the $N$ samples exactly, and a time shift $z_{n-s}$ only multiplies $c_k$ by the phase $e^{-2\pi i k s/N}$ (the radii do not change, only the starting point). A circle traced once needs a single term ($k = 1$), and the heart $x = 16\sin^3 s$, $y = 13\cos s - 5\cos 2s - 2\cos 3s - \cos 4s$ is a trigonometric polynomial, so exactly 8 circles ($k = \pm1 \dots \pm4$) draw it perfectly. Polygons are different: at every corner the velocity jumps, so $|c_k|$ falls only like $1/k^2$ and the error with $M$ circles like $M^{-3}$; with few circles the corners are rounded off and ringing wiggles run along the edges. The curve itself is continuous, so (unlike the square wave of the Fourier series problem, where the function jumps) the wiggles shrink to nothing; the 9 % Gibbs overshoot lives in the velocity, which does jump. Keeping the biggest circles first makes the error fall monotonically, because by Parseval the error is exactly $\sum|c_k|^2$ over the dropped circles. History: Ptolemy's astronomy (Almagest, 2nd century) moved planets on a circle (epicycle) whose centre rides on another circle (deferent); with enough epicycles any periodic closed path can be matched, which is why the system fitted observations so well for 1400 years without being the right physics. The logo path is the hexagon outline of the Tensorizer cube plus its three inner edges, walked out and back. Try the star with few circles.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/fourier_epicycles/main.py
"""Main loop: closed curve → DFT → circles sorted by radius → chains with more and more circles → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    name, n = params["shape"], params["n_points"]
    z = tensors.shape(name, n, dev)  # (N,) complex
    m_stages = solve.stages(n, params["max_terms"])
    if fast_mode():
        m_stages = m_stages[:2]
    res = solve.analyse(z, m_stages)
    print(f"{name}: N = {n} samples z_n = x_n + i y_n, frequencies k = {res['k'][0].item()} … {res['k'][-1].item()}")
    print(f"Parseval: (1/N) Σ|z_n|² = {res['power']:.12f},  Σ|c_k|² = {res['coef_power']:.12f}")
    top = res["order"][:6]
    print("largest circles: " + ", ".join(f"k = {k:+d} r = {r:.3f}" for k, r in
          zip(res["k"][top].tolist(), res["c"][top].abs().tolist())))  # fmt: skip
    err = res["err"]
    for m in m_stages:
        e, p = err[m - 1].item(), res["parseval"][m - 1].item()
        print(f"  M = {m:4d} circles: mean squared error {e:.3e}  (Parseval {p:.3e})")
    mono = bool((err[1:] <= err[:-1] + 1e-12).all())
    print(f"error falls monotonically with M: {mono};  all {n} circles: error {err[-1].item():.1e}")
    return plot.animate(z, res, m_stages, name, out_dir / "fourier_epicycles.gif",
                        frames_per_stage=4 if fast_mode() else 12)  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/fourier_epicycles/equation.py
r"""Epicycles → tensor translation: the complex discrete Fourier transform of a closed curve.

Samples z_n (n = 0 … N−1) at times t_n = n/N; frequencies k = −N/2 … N/2 − 1 (integers, the turns per trip).

    W_{kn} = e^{-2\pi i k n / N},        c_k = \tfrac1N W_{kn} z_n                       (n summed)    the DFT
    z(t) = c_k e^{2\pi i k t}                                                           (k summed)    the inverse

Each term c_k e^{2πikt} is a circle of radius |c_k| turning k times per trip, starting at angle arg c_k; adding them
tip to tail is a chain of epicycles. Keep only the M largest circles (rank r_k < M, sorted by radius):

    S_{mt} = [r_k < M_m]\, c_k\, e^{2\pi i k t}                                         (k summed)
    E_m = \tfrac1N |z_n - S_{mn}|^2 = \sum_{r_k \ge M_m} |c_k|^2                          (Parseval)
"""

import math

import torch


def frequencies(n: int, device: torch.device) -> torch.Tensor:
    """k = −N/2 … N/2 − 1 (N,) int64: negative k turn clockwise, positive anticlockwise."""
    return torch.arange(n, device=device) - n // 2


def dft(z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """c_k = (1/N) W_{kn} z_n with W_{kn} = exp(−2πi k n / N).  z (N,) complex → c (N,) complex, k (N,). n summed."""
    n = z.shape[0]
    k = frequencies(n, z.device)
    idx = torch.arange(n, device=z.device)
    W = torch.exp(-2j * math.pi * torch.outer(k, idx).to(torch.float64) / n)  # (N, N) the DFT matrix
    return torch.einsum("kn,n->k", W, z) / n, k


def idft(c: torch.Tensor, k: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """z(t) = c_k exp(2πi k t) at any times t (T,) → (T,) complex.  k summed."""
    E = torch.exp(2j * math.pi * torch.outer(t, k.to(torch.float64)))  # (T, K) one rotating unit vector per term
    return torch.einsum("tk,k->t", E, c)


def by_radius(c: torch.Tensor) -> torch.Tensor:
    """Indices of the terms from the largest circle |c_k| to the smallest (N,) int64."""
    return torch.argsort(c.abs(), descending=True, stable=True)


def partial_sums(c: torch.Tensor, k: torch.Tensor, order: torch.Tensor, m: torch.Tensor, t: torch.Tensor):
    """S_{mt}: the curve drawn by only the M_m largest circles, for every M in m (M,) at once → (M, T) complex.

    keep_{mk} = [rank of term k < M_m] is a 0/1 matrix; S = keep · (c_k e^{2πikt}), k summed.
    """
    rank = torch.empty_like(order)
    rank[order] = torch.arange(order.shape[0], device=order.device)  # (K,) rank of each term by radius
    keep = (rank[None, :] < m[:, None]).to(c.dtype)  # (M, K)
    E = torch.exp(2j * math.pi * torch.outer(t, k.to(torch.float64)))  # (T, K)
    return torch.einsum("mk,k,tk->mt", keep, c, E)


def chain(c: torch.Tensor, k: torch.Tensor, order: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Centres of the epicycles at times t: (T, K + 1) complex, from 0 (the fixed centre) to the pen tip.

    Term j of the chain is c_{o_j} e^{2πi k_{o_j} t}, biggest first; the running sum gives each circle's centre.
    """
    terms = c[order] * torch.exp(2j * math.pi * torch.outer(t, k[order].to(torch.float64)))  # (T, K)
    return torch.cat([terms.new_zeros(t.shape[0], 1), torch.cumsum(terms, dim=1)], dim=1)


def mean_power(z: torch.Tensor) -> torch.Tensor:
    """(1/N) Σ_n |z_n|², which Parseval says equals Σ_k |c_k|²."""
    return (z.abs() ** 2).mean()


def sample_error(z: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    """E_m = (1/N) Σ_n |z_n − S_{mn}|²: mean squared distance at the sample points, (M, N) → (M,)."""
    return ((s - z).abs() ** 2).mean(-1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/fourier_epicycles/tensors.py
"""Build the concrete tensors: a closed curve sampled at N points (equal time steps) as complex z_n = x_n + i y_n.

Three original shapes:
  logo   the outline of the Tensorizer logo (frontend/public/logo-light.svg): the hexagon silhouette of a 3×3×3 cube
         plus the three inner edges (the "Y"), walked as one closed pen stroke that goes out and back along each spoke
  star   a five-pointed star polygon (sharp corners: ringing when few circles are kept)
  heart  the smooth parametric heart x = 16 sin³s, y = 13 cos s − 5 cos 2s − 2 cos 3s − cos 4s, sampled evenly in s:
         a trigonometric polynomial (16 sin³s = 12 sin s − 4 sin 3s), so exactly 8 circles k = ±1 … ±4 draw it
"""

import math

import torch

SHAPES = ["logo", "star", "heart"]

# corners of the logo's cube in its 64 × 64 SVG frame (y pointing down), taken from logo-light.svg
_TOP, _UR, _LR = (32.00, 2.45), (57.59, 17.68), (57.59, 46.77)
_BOT, _LL, _UL, _MID = (32.00, 61.33), (6.41, 46.77), (6.41, 17.68), (32.00, 32.00)
LOGO_PATH = [_UL, _TOP, _UR, _LR, _BOT, _LL, _UL, _MID, _UR, _MID, _BOT, _MID]  # closes back to _UL


def times(n: int, device: torch.device) -> torch.Tensor:
    """t_n = n / N, n = 0 … N−1: one trip round the curve takes t from 0 to 1.  (N,) float64."""
    return torch.arange(n, dtype=torch.float64, device=device) / n


def resample(corners: torch.Tensor, n: int) -> torch.Tensor:
    """N points equally spaced by arc length along the closed polygon through `corners` (P,) complex → (N,) complex."""
    seg = torch.roll(corners, -1) - corners  # (P,) segment vectors, the last one closes the loop
    length = seg.abs()
    start = torch.cat([length.new_zeros(1), torch.cumsum(length, 0)[:-1]])  # (P,) arc length at each corner
    s = torch.arange(n, dtype=torch.float64, device=corners.device) * length.sum() / n  # (N,) target arc lengths
    which = torch.searchsorted(start, s, right=True) - 1  # (N,) segment that holds each point
    frac = (s - start[which]) / length[which]
    return corners[which] + frac * seg[which]


def normalise(z: torch.Tensor) -> torch.Tensor:
    """Centre on the mean and scale so the farthest point sits on the unit circle."""
    z = z - z.mean()
    return z / z.abs().max()


def shape(name: str, n: int, device: torch.device) -> torch.Tensor:
    """The curve `name` sampled at N points, (N,) complex128, centred and inside the unit disc."""
    if name == "logo":
        xy = torch.tensor(LOGO_PATH, dtype=torch.float64, device=device)
        z = resample(torch.complex(xy[:, 0], -xy[:, 1]), n)  # flip y: SVG y points down
    elif name == "star":
        a = math.pi / 2 + torch.arange(10, dtype=torch.float64, device=device) * math.pi / 5  # outer, inner, …
        r = torch.where(torch.arange(10, device=device) % 2 == 0, 1.0, 0.38).to(torch.float64)
        z = resample(r * torch.exp(1j * a), n)
    elif name == "heart":
        s = 2 * math.pi * times(n, device)  # sampled evenly in the parameter s, not in arc length
        x = 16 * torch.sin(s) ** 3
        y = 13 * torch.cos(s) - 5 * torch.cos(2 * s) - 2 * torch.cos(3 * s) - torch.cos(4 * s)
        z = torch.complex(x, y)
    else:
        raise ValueError(f"unknown shape {name!r}; choose from {SHAPES}")
    return normalise(z)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/fourier_epicycles/solve.py
"""The solver: one DFT, then every truncated chain of circles at once."""

import torch

from . import equation, tensors


def stages(n: int, max_terms: int) -> list[int]:
    """How many circles each stage of the animation keeps: 1, 2, 4, 8, … up to max_terms (and at most N)."""
    top = min(n, max_terms)
    m, out = 1, []
    while m < top:
        out.append(m)
        m *= 2
    return out + [top]


def analyse(z: torch.Tensor, m_stages: list[int], n_dense: int = 600) -> dict:
    """DFT of the samples, the error with every number of circles M = 1 … N, and the curves of the stages.

    Returns c, k, order (biggest circle first), err (N,) for M = 1 … N, the Parseval prediction (N,),
    the stage curves S (stages, n_dense) on a fine time grid, and the energy check.
    """
    n = z.shape[0]
    c, k = equation.dft(z)
    order = equation.by_radius(c)
    t = tensors.times(n, z.device)
    m_all = torch.arange(1, n + 1, device=z.device)
    err = equation.sample_error(z, equation.partial_sums(c, k, order, m_all, t))  # (N,) for M = 1 … N
    energy = c[order].abs() ** 2  # (N,) biggest first
    left_out = energy.sum() - torch.cumsum(energy, 0)  # (N,) energy of the circles NOT kept with M = 1 … N
    t_dense = tensors.times(n_dense, z.device)
    curves = equation.partial_sums(c, k, order, torch.tensor(m_stages, device=z.device), t_dense)  # (stages, T)
    return {
        "c": c,
        "k": k,
        "order": order,
        "err": err,
        "parseval": left_out.clamp_min(0),
        "curves": curves,
        "t_dense": t_dense,
        "power": equation.mean_power(z).item(),
        "coef_power": energy.sum().item(),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/fourier_epicycles/plot.py
"""Animate the chain of rotating circles drawing the curve, with more circles at every stage, beside the error curve."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, save_gif

from . import equation

MAX_DRAWN = 40  # circles drawn in the chain (the rest are too small to see)


def _circles(centres: np.ndarray, radii: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """All circles as one polyline, separated by NaN, so a single Line2D draws them."""
    a = np.linspace(0, 2 * np.pi, 49)
    xs = centres.real[:, None] + radii[:, None] * np.cos(a)  # (C, 49)
    ys = centres.imag[:, None] + radii[:, None] * np.sin(a)
    pad = np.full((len(radii), 1), np.nan)
    return np.hstack([xs, pad]).ravel(), np.hstack([ys, pad]).ravel()


def animate(z: torch.Tensor, res: dict, m_stages: list[int], name: str, path: Path, frames_per_stage: int = 12):
    target = z.cpu().numpy()
    c, k, order = res["c"], res["k"], res["order"]
    radii = c[order].abs().cpu().numpy()
    curves = res["curves"].cpu().numpy()  # (stages, T)
    n_dense = curves.shape[1]
    err, parseval = res["err"].cpu().numpy(), res["parseval"].cpu().numpy()
    n = len(err)

    t_frames = torch.linspace(0, 1, frames_per_stage, dtype=torch.float64, device=c.device)
    jobs = [(s, f) for s in range(len(m_stages)) for f in range(frames_per_stage)]
    chains = {}  # stage → (frames, M + 1) circle centres at the frame times

    fig, (ax, ax_err) = plt.subplots(1, 2, figsize=(8.6, 4.3), gridspec_kw={"width_ratios": [1.05, 1]})
    closed = np.append(target, target[0])
    ax.plot(closed.real, closed.imag, color="#d1d5db", lw=3, zorder=0)
    (circ_line,) = ax.plot([], [], color="#9ca3af", lw=0.7)
    (arm_line,) = ax.plot([], [], color=ACCENT[0], lw=1.2, marker="o", ms=1.5)
    (trace,) = ax.plot([], [], color=ACCENT[1], lw=1.8)
    (pen,) = ax.plot([], [], "o", color=ACCENT[1], ms=4)
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    title = ax.set_title("")

    m_axis = np.arange(1, n + 1)
    ok = m_axis < n  # with all N circles the error is 0 (round-off): left off the log axis
    ax_err.loglog(m_axis[ok], err[ok], color=ACCENT[0], lw=2, label="error at the samples")
    ax_err.loglog(m_axis[ok][::4], parseval[ok][::4], "o", ms=3, mfc="none", color=ACCENT[2],
                  label=r"Parseval: $\Sigma\,|c_k|^2$ of dropped circles")  # fmt: skip
    ax_err.set_ylim(max(err[ok].min() * 0.3, 1e-12), err[0] * 3)
    ref = m_axis[3 : n // 2].astype(float)
    if name != "heart":
        anchor = err[7] * (ref / 8.0) ** -3
        ax_err.loglog(ref, anchor, "--", color="#9ca3af", lw=1, label=r"$\propto M^{-3}$ (corners: $|c_k|\sim k^{-2}$)")
    ax_err.set_xlabel("circles kept $M$ (largest first)")
    ax_err.set_ylabel(r"mean $|z_n - S_M(t_n)|^2$")
    ax_err.set_title("reconstruction error")
    ax_err.legend(fontsize=7, loc="lower left")
    marker = ax_err.axvline(1, color="#f59e0b", lw=1.5)

    def update(job: int):
        s, f = jobs[job]
        m = m_stages[s]
        if s not in chains:
            chains[s] = equation.chain(c, k, order[:m], t_frames).cpu().numpy()  # (frames, M + 1)
        centres = chains[s][f]
        shown = min(m, MAX_DRAWN)
        cx, cy = _circles(centres[:shown], radii[:shown])
        circ_line.set_data(cx, cy)
        arm_line.set_data(centres[: shown + 1].real, centres[: shown + 1].imag)
        upto = int(round(f / (frames_per_stage - 1) * (n_dense - 1))) + 1
        trace.set_data(curves[s, :upto].real, curves[s, :upto].imag)
        pen.set_data([centres[-1].real], [centres[-1].imag])
        title.set_text(f"{name}: {m} circle{'s' if m > 1 else ''}, error {err[m - 1]:.1e}")
        marker.set_xdata([m, m])

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(jobs))), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'shape': 'logo',
    'n_points': 512,
    'max_terms': 256
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.fourier_epicycles")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.fourier_epicycles.main")
problem_dir = Path("problems/maths/fourier_epicycles")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
